In [5]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nymeria_gaze_tools as ngt

DATA_ROOT      = Path("../data/processed")
SUMMARIES_PATH = DATA_ROOT / "all_session_summaries.csv"
FIGURES_DIR    = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

print("Setup complete.")


Setup complete.


In [6]:
# Load enriched metadata
catalog_raw = ngt.load_metadata(data_root=DATA_ROOT)

# --- Standard filter block ---
PAIRED_SESSIONS = "exclude"  # "exclude" = Option A,  "include" = Option B

catalog = catalog_raw.copy()
catalog = catalog[catalog["sampling_rate_hz"] == 10]          # drop 30 Hz
catalog = catalog[catalog["noise_flag"] == False]             # drop bad trim sessions
catalog = catalog[catalog["has_gaze_data"] == True]           # must have gaze data
if PAIRED_SESSIONS == "exclude":
    catalog = catalog[catalog["has_two_participants"] == False]
catalog = (catalog
           .sort_values("date")
           .drop_duplicates(["fake_name", "script"])
           .reset_index(drop=True))

print(f"Raw sessions:      {len(catalog_raw)}")
print(f"Filtered sessions: {len(catalog)}")
print(f"Unique participants: {catalog['fake_name'].nunique()}")


Raw sessions:      1100
Filtered sessions: 810
Unique participants: 193


In [7]:
c = catalog_raw.copy()

c1 = c[c["sampling_rate_hz"] == 10]
c2 = c1[c1["noise_flag"] == False]
c3 = c2[c2["has_gaze_data"] == True]
c4 = c3[c3["has_two_participants"] == False]
c5 = c4.sort_values("date").drop_duplicates(["fake_name", "script"])

print(f"Start:                    {len(c)}")
print(f"After 30 Hz drop:         {len(c1)}  (-{len(c)  - len(c1)})")
print(f"After noise flag drop:    {len(c2)}  (-{len(c1) - len(c2)})")
print(f"After no-gaze drop:       {len(c3)}  (-{len(c2) - len(c3)})")
print(f"After paired drop:        {len(c4)}  (-{len(c3) - len(c4)})")
print(f"After dedup (first only): {len(c5)}  (-{len(c4) - len(c5)})")


Start:                    1100
After 30 Hz drop:         1066  (-34)
After noise flag drop:    1064  (-2)
After no-gaze drop:       1064  (-0)
After paired drop:        847  (-217)
After dedup (first only): 810  (-37)


In [8]:
# looking at the sessions we dropped as duplicates
c4 = catalog_raw.copy()
c4 = c4[c4["sampling_rate_hz"] == 10]
c4 = c4[c4["noise_flag"] == False]
c4 = c4[c4["has_gaze_data"] == True]
c4 = c4[c4["has_two_participants"] == False]

dropped_dupes = (c4.sort_values("date")
                   .groupby(["fake_name", "script"])
                   .filter(lambda g: len(g) > 1))

role_check = (dropped_dupes.groupby(["fake_name", "script"])
                            .agg(n_sessions=("session_id", "count"),
                                 session_ids=("session_id", list))
                            .reset_index())
role_check["both_roles"] = role_check["session_ids"].apply(lambda x: "s0" in x and "s1" in x)

print(f"Participant-activity pairs with duplicates: {len(role_check)}")
print(f"Did both roles (s0 + s1): {role_check['both_roles'].sum()}")
print(f"Repeated same role:       {(~role_check['both_roles']).sum()}")
role_check.sort_values("script")


Participant-activity pairs with duplicates: 36
Did both roles (s0 + s1): 0
Repeated same role:       36


,fake_name,script,n_sessions,session_ids,both_roles
16,dylan_jones,S10-Housekeeping,2,"[s1, s1]",False
9,brooke_butler,S10-Housekeeping,2,"[s1, s1]",False
4,arthur_byrd,S12-Game_night,2,"[s0, s0]",False
21,jeremy_lewis,S16-Simon_says,2,"[s0, s0]",False
33,tasha_lee,S16-Simon_says,2,"[s0, s0]",False
32,suzanne_romero,S16-Simon_says,2,"[s0, s0]",False
18,george_james,S16-Simon_says,2,"[s0, s0]",False
34,xavier_norris,S16-Simon_says,2,"[s1, s1]",False
20,holly_keller,S16-Simon_says,2,"[s0, s0]",False
25,megan_mejia,S16-Simon_says,2,"[s1, s1]",False


In [9]:
# option A: including only cooking session where there is only one participant
# option B: including all sessions

base = catalog_raw.copy()
base = base[base["sampling_rate_hz"] == 10]
base = base[base["noise_flag"] == False]
base = base[base["has_gaze_data"] == True]
base = base[base["script"] == "S7-Cooking"]

option_a = (base[base["has_two_participants"] == False]
            .sort_values("date")
            .drop_duplicates("fake_name")
            .reset_index(drop=True))

option_b = (base
            .sort_values("date")
            .drop_duplicates("fake_name")
            .reset_index(drop=True))

print(f"Option A (solo only):    {len(option_a)} sessions, {option_a['fake_name'].nunique()} participants")
print(f"Option B (all cooking):  {len(option_b)} sessions, {option_b['fake_name'].nunique()} participants")


Option A (solo only):    97 sessions, 97 participants
Option B (all cooking):  128 sessions, 128 participants


In [10]:
summaries = pd.read_csv(SUMMARIES_PATH)

METRICS = ["mean_yaw_deg", "var_yaw_deg", "mean_pitch_deg", "var_pitch_deg", "mean_depth_m", "var_depth_m"]

cooking_a = summaries[summaries["sequence_uid"].isin(option_a["sequence_uid"])]
cooking_b = summaries[summaries["sequence_uid"].isin(option_b["sequence_uid"])]

print("Metric              Option A (solo)        Option B (all)")
print("-" * 60)
for m in METRICS:
    a_mean = cooking_a[m].mean()
    b_mean = cooking_b[m].mean()
    print(f"{m:<20}  {a_mean:+.3f}              {b_mean:+.3f}")


Metric              Option A (solo)        Option B (all)
------------------------------------------------------------
mean_yaw_deg          -0.123              -0.206
var_yaw_deg           +121.839              +123.871
mean_pitch_deg        -23.647              -23.342
var_pitch_deg         +121.172              +121.673
mean_depth_m          +0.982              +1.005
var_depth_m           +0.786              +0.821


In [11]:
def load_and_preprocess(uids, catalog, data_root):
    dfs = []
    for uid in uids:
        row = catalog[catalog["sequence_uid"] == uid].iloc[0]
        raw = ngt.load_session(uid, data_root=data_root)
        df  = ngt.preprocess(
            raw,
            trim_start_min=row["trim_start_sec"] / 60,
            trim_end_min=row["trim_end_sec"] / 60,
        )
        dfs.append(df)
    return dfs

print("Loading Option B sessions (superset)...")
dfs_b = load_and_preprocess(option_b["sequence_uid"], catalog_raw, DATA_ROOT)

# Option A is a subset of B
dfs_a = [df for uid, df in zip(option_b["sequence_uid"], dfs_b)
         if uid in set(option_a["sequence_uid"])]

print(f"Option A: {len(dfs_a)} sessions loaded")
print(f"Option B: {len(dfs_b)} sessions loaded")


Loading Option B sessions (superset)...
Option A: 96 sessions loaded
Option B: 128 sessions loaded


In [12]:
for dfs, label in [(dfs_a, "Option A (solo)"), (dfs_b, "Option B (all)")]:
    fig = ngt.plot_population_density(dfs, title=f"Cooking — {label}  (n={len(dfs)} participants)")
    fig.show()

# Spatial spread
print("\nSpatial spread (mean per-session variance across participants):")
print(f"{'':25} Option A    Option B")
for col, label in [("avg_yaw_deg", "Yaw variance"), ("pitch_deg", "Pitch variance")]:
    var_a = np.mean([df[col].var() for df in dfs_a])
    var_b = np.mean([df[col].var() for df in dfs_b])
    print(f"  {label:<23} {var_a:.2f}       {var_b:.2f}")



Spatial spread (mean per-session variance across participants):
                          Option A    Option B
  Yaw variance            112.22       114.72
  Pitch variance          116.44       116.88


In [13]:
PAIRED_ACTIVITIES = ["S20-Party", "S7-Cooking", "S10-Housekeeping", 
                     "S3-Welcome_to_my_place", "S12-Game_night", 
                     "S8-Having_a_meal", "S19-Fresh_air"]

paired = catalog_raw[
    catalog_raw["script"].isin(PAIRED_ACTIVITIES) &
    (catalog_raw["sampling_rate_hz"] == 10) &
    (catalog_raw["noise_flag"] == False) &
    (catalog_raw["has_gaze_data"] == True)
]

print(f"Total sessions to load: {len(paired)}")
print()
print(paired.groupby("script").size().sort_values(ascending=False))


Total sessions to load: 581

script
S7-Cooking                154
S12-Game_night            109
S10-Housekeeping           79
S20-Party                  67
S19-Fresh_air              61
S3-Welcome_to_my_place     61
S8-Having_a_meal           50
dtype: int64


In [14]:
all_dfs = {}

print(f"Loading {len(paired)} sessions...")
for i, (_, row) in enumerate(paired.iterrows()):
    if i % 50 == 0:
        print(f"  {i}/{len(paired)}")
    raw = ngt.load_session(row["sequence_uid"], data_root=DATA_ROOT)
    all_dfs[row["sequence_uid"]] = ngt.preprocess(
        raw,
        trim_start_min=row["trim_start_sec"] / 60,
        trim_end_min=row["trim_end_sec"] / 60,
    )

print(f"Done. {len(all_dfs)} sessions loaded.")


Loading 581 sessions...
  0/581
  50/581
  100/581
  150/581
  200/581
  250/581
  300/581
  350/581
  400/581
  450/581
  500/581
  550/581
Done. 581 sessions loaded.


In [15]:
# Function to repeat the option A vs option B analysis for other two-participant activities
def compare_options(script, catalog_raw, all_dfs):
    base = catalog_raw[
        (catalog_raw["script"] == script) &
        (catalog_raw["sampling_rate_hz"] == 10) &
        (catalog_raw["noise_flag"] == False) &
        (catalog_raw["has_gaze_data"] == True)
    ]

    opt_a = (base[base["has_two_participants"] == False]
             .sort_values("date")
             .drop_duplicates("fake_name"))

    opt_b = (base.sort_values("date")
                 .drop_duplicates("fake_name"))

    dfs_a = [all_dfs[uid] for uid in opt_a["sequence_uid"] if uid in all_dfs]
    dfs_b = [all_dfs[uid] for uid in opt_b["sequence_uid"] if uid in all_dfs]

    print(f"\n{'='*50}")
    print(f"{script}")
    print(f"Option A (solo): {len(dfs_a)} participants")
    print(f"Option B (all):  {len(dfs_b)} participants")

    # Spatial spread
    print(f"\n{'':25} Option A    Option B")
    for col, label in [("avg_yaw_deg", "Yaw variance"), ("pitch_deg", "Pitch variance")]:
        var_a = np.mean([df[col].var() for df in dfs_a])
        var_b = np.mean([df[col].var() for df in dfs_b])
        print(f"  {label:<23} {var_a:.2f}       {var_b:.2f}")

    # Heatmaps
    for dfs, label in [(dfs_a, "Option A (solo)"), (dfs_b, "Option B (all)")]:
        fig = ngt.plot_population_density(dfs, title=f"{script} — {label}  (n={len(dfs)})")
        fig.show()


In [16]:
for script in PAIRED_ACTIVITIES:
    compare_options(script, catalog_raw, all_dfs)


S20-Party
Option A (solo): 13 participants
Option B (all):  46 participants

                          Option A    Option B
  Yaw variance            100.25       102.44
  Pitch variance          144.84       146.85



S7-Cooking
Option A (solo): 97 participants
Option B (all):  128 participants

                          Option A    Option B
  Yaw variance            112.25       114.72
  Pitch variance          115.92       116.88



S10-Housekeeping
Option A (solo): 54 participants
Option B (all):  77 participants

                          Option A    Option B
  Yaw variance            134.90       133.68
  Pitch variance          123.33       126.43



S3-Welcome_to_my_place
Option A (solo): 39 participants
Option B (all):  61 participants

                          Option A    Option B
  Yaw variance            186.11       184.58
  Pitch variance          110.38       112.57



S12-Game_night
Option A (solo): 89 participants
Option B (all):  107 participants

                          Option A    Option B
  Yaw variance            87.69       91.97
  Pitch variance          91.73       97.91



S8-Having_a_meal
Option A (solo): 46 participants
Option B (all):  50 participants

                          Option A    Option B
  Yaw variance            142.73       141.83
  Pitch variance          159.20       156.80



S19-Fresh_air
Option A (solo): 55 participants
Option B (all):  60 participants

                          Option A    Option B
  Yaw variance            116.84       116.53
  Pitch variance          118.70       118.14


What we did:

We took the full Nymeria dataset of 1,100 sessions and figured out the cleanest possible dataset to use for analysis. We applied four filters — dropping sessions recorded at 30 Hz (different sampling rate), dropping sessions with suspiciously long setup times (flagged as noise outliers using mean + 3 SD), keeping only each person's first recording when they did the same activity twice, and excluding sessions where two participants were recorded together. This left us with 810 sessions from 193 participants as our clean analysis dataset.

We also investigated whether excluding those two-participant sessions was actually necessary, by comparing gaze behavior with and without them across all seven affected activities.

What we found:

1. The data cleaning removed 290 sessions — most of them (217) were two-participant sessions, 34 were 30 Hz sessions, 37 were duplicate recordings by the same person, and only 3 were flagged for bad timing data.

2. Every participant who did the same activity twice had the same session_id — nobody switched roles between recordings. This means session_id field (s0/s1) is not a role indicator.

3. Having a co-participant doesn't change gaze behavior — across all seven activities (cooking, party, housekeeping, game night, etc.), the gaze distributions were nearly identical whether we included or excluded paired sessions. The differences were less than 5° of variance in every case.

4. Party is the one exception — only 13 solo Party sessions exist, which is too few for reliable analysis. Most Party sessions had two participants, so Party should either be analyzed with Option B or flagged as low-n.


In [17]:
# Gaze type breakdown
gaze_per_participant = catalog_raw.groupby("fake_name")["gaze_type"].apply(set)

only_personalized = (gaze_per_participant == {"personalized"}).sum()
only_general      = (gaze_per_participant == {"general"}).sum()
both              = (gaze_per_participant.apply(len) > 1).sum()

print(f"Participants with only personalized gaze: {only_personalized}")
print(f"Participants with only general gaze:      {only_general}")
print(f"Participants with both:                   {both}")

# Who are the general-only participants?
general_only_names = gaze_per_participant[gaze_per_participant == {"general"}].index.tolist()
print(f"\nGeneral-only participants and their sessions:")
catalog_raw[catalog_raw["fake_name"].isin(general_only_names)][["fake_name","script","gaze_type"]]


Participants with only personalized gaze: 164
Participants with only general gaze:      4
Participants with both:                   68

General-only participants and their sessions:


,fake_name,script,gaze_type
635,justin_martin,S16-Simon_says,general
636,justin_martin,S19-Fresh_air,general
637,justin_martin,S18-Hike,general
750,mark_richardson,S7-Cooking,general
751,mark_richardson,S8-Having_a_meal,general
752,mark_richardson,S3-Welcome_to_my_place,general
753,mark_richardson,S2-Where_is_X,general
754,mark_richardson,S12-Game_night,general
887,ronald_harris,S3-Welcome_to_my_place,general
888,ronald_harris,S2-Where_is_X,general


In [18]:
catalog_raw = ngt.load_metadata(data_root=DATA_ROOT)

c = catalog_raw.copy()
print(f"Start:                          {len(c):>5} sessions, {c['fake_name'].nunique():>3} participants")

c = c[c["sampling_rate_hz"] == 10]
print(f"After dropping 30 Hz:           {len(c):>5} sessions, {c['fake_name'].nunique():>3} participants")

c = c[c["noise_flag"] == False]
print(f"After noise flag:               {len(c):>5} sessions, {c['fake_name'].nunique():>3} participants")

c = c[c["has_two_participants"] == False]
print(f"After excluding paired:         {len(c):>5} sessions, {c['fake_name'].nunique():>3} participants")

c = (c.sort_values("date")
      .drop_duplicates(["fake_name", "script"])
      .reset_index(drop=True))
print(f"After deduplication:            {len(c):>5} sessions, {c['fake_name'].nunique():>3} participants")


Start:                           1100 sessions, 236 participants
After dropping 30 Hz:            1066 sessions, 228 participants
After noise flag:                1064 sessions, 228 participants
After excluding paired:           847 sessions, 193 participants
After deduplication:              810 sessions, 193 participants
